In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

2026-03-11 16:52:11,359 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-11 16:52:11,360 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

In [8]:
with get_session(engine) as session:                                                                                                 
    results = session.query(cultpass.Experience).all()                                                                               
    for r in results:                                                                                                                
        print(r)   

<Experience(experience_id='f4b88f', title='Carnival History Tour in Olinda', when='2026-03-12 17:10:56.909221')>
<Experience(experience_id='f9772c', title='Sunset Paddleboarding', when='2026-03-13 17:10:56.923770')>
<Experience(experience_id='a5c952', title='Pelourinho Colonial Walk', when='2026-03-14 17:10:56.923818')>
<Experience(experience_id='aa29e9', title='Samba Night at Lapa', when='2026-03-15 17:10:56.923839')>
<Experience(experience_id='e9c2c6', title='Christ the Redeemer Experience', when='2026-03-16 17:10:56.923855')>
<Experience(experience_id='308907', title='Modern Art at MASP', when='2026-03-17 17:10:56.923871')>
<Experience(experience_id='c0d4c0', title='Ibirapuera Park Bike Ride', when='2026-03-18 17:10:56.923886')>


**User**

In [9]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [10]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [11]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

In [12]:
with get_session(engine) as session:                                                                                                 
    results = session.query(cultpass.User).all()                                                                               
    for r in results:                                                                                                                
        print(r)   

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


**Subscription**

In [13]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

In [15]:
with get_session(engine) as session:                                                                                                 
    results = session.query(cultpass.Subscription).all()                                                                               
    for r in results:                                                                                                                
        print(r)  

<Subscription(subscription_id='713918', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='51035a', user_id='f556c0', status='cancelled', tier='premium')>
<Subscription(subscription_id='198fea', user_id='88382b', status='active', tier='premium')>
<Subscription(subscription_id='e42e7b', user_id='888fb2', status='active', tier='premium')>
<Subscription(subscription_id='938598', user_id='f1f10d', status='cancelled', tier='premium')>
<Subscription(subscription_id='e4a6ce', user_id='e6376d', status='active', tier='premium')>


**Reservation**

In [16]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [17]:
with get_session(engine) as session:                                                                                                 
    results = session.query(cultpass.Reservation).all()                                                                               
    for r in results:                                                                                                                
        print(r)  

<Reservation(reservation_id='0a51ac', user_id='a4ab87', experience_id='f4b88f', status='reserved')>
<Reservation(reservation_id='06746f', user_id='a4ab87', experience_id='e9c2c6', status='reserved')>


In [18]:
# DONE: Add more data
# Please notice that the reservations were set to first user only 
# If you want to simulate more users later, please create more reservations per user
with get_session(engine) as session:
    reservations = []

    for idx, user in enumerate(cultpass_users):
        res = cultpass.Reservation(
            reservation_id=str(uuid.uuid4())[:6],
            user_id=user["id"],
            experience_id=random.choice(experience_ids),
            status="reserved",
        )
        reservations.append(res)

    session.add_all(reservations)

In [19]:
with get_session(engine) as session:                                                                                                 
    results = session.query(cultpass.Reservation).all()                                                                               
    for r in results:                                                                                                                
        print(r)  

<Reservation(reservation_id='0a51ac', user_id='a4ab87', experience_id='f4b88f', status='reserved')>
<Reservation(reservation_id='06746f', user_id='a4ab87', experience_id='e9c2c6', status='reserved')>
<Reservation(reservation_id='5c5bd7', user_id='a4ab87', experience_id='f4b88f', status='reserved')>
<Reservation(reservation_id='b76981', user_id='f556c0', experience_id='e9c2c6', status='reserved')>
<Reservation(reservation_id='9a64b5', user_id='88382b', experience_id='a5c952', status='reserved')>
<Reservation(reservation_id='08b557', user_id='888fb2', experience_id='c0d4c0', status='reserved')>
<Reservation(reservation_id='705d06', user_id='f1f10d', experience_id='f4b88f', status='reserved')>
<Reservation(reservation_id='8033de', user_id='e6376d', experience_id='308907', status='reserved')>


# Tests

In [20]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [21]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='713918', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='51035a', user_id='f556c0', status='cancelled', tier='premium')>
<Subscription(subscription_id='198fea', user_id='88382b', status='active', tier='premium')>
<Subscription(subscription_id='e42e7b', user_id='888fb2', status='active', tier='premium')>
<Subscription(subscription_id='938598', user_id='f1f10d', status='cancelled', tier='premium')>
<Subscription(subscription_id='e4a6ce', user_id='e6376d', status='active', tier='premium')>


In [22]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='f4b88f', title='Carnival History Tour in Olinda', when='2026-03-12 17:10:56.909221')>
<Experience(experience_id='f9772c', title='Sunset Paddleboarding', when='2026-03-13 17:10:56.923770')>
<Experience(experience_id='a5c952', title='Pelourinho Colonial Walk', when='2026-03-14 17:10:56.923818')>
<Experience(experience_id='aa29e9', title='Samba Night at Lapa', when='2026-03-15 17:10:56.923839')>
<Experience(experience_id='e9c2c6', title='Christ the Redeemer Experience', when='2026-03-16 17:10:56.923855')>
<Experience(experience_id='308907', title='Modern Art at MASP', when='2026-03-17 17:10:56.923871')>
<Experience(experience_id='c0d4c0', title='Ibirapuera Park Bike Ride', when='2026-03-18 17:10:56.923886')>
